# **MultiModal RAG App for Video Processing With LlamaIndex and LanceDB**

### 1. llamaindex framework
### 2. Lancedb Vector DataBase
### 3. LLM MultiModAl GPT-4V or Google-gemini-pro-vision


# **Steps Need to follow:**
#### 1. Download video from YouTube, process and store it.

#### 2. Build Multi-Modal index and vector store for both texts and images.

#### 3. Retrieve relevant images and context, use both to augment the prompt.

#### 4. Using GPT4V for reasoning the correlations between the input query and augmented data and generating final response.

YouTube Video
   ↓
Video Download
   ↓
Frames (Images) + Audio/Text
   ↓
CLIP Embeddings (Text + Image)
   ↓
LanceDB Vector Store (Multi-modal index)
   ↓
Retriever (Text + Image similarity)
   ↓
Prompt Augmentation (Context + Images)
   ↓
GPT-4V / Gemini Vision (Reasoning + Answer)


In [92]:
%pip install llama-index
%pip install llama-index-vector-stores-lancedb
%pip install llama-index-multi-modal-llms-openai
%pip install llama-index-embeddings-clip
%pip install llama-index-readers-file
%pip install lancedb
%pip install pytube opencv-python pillow moviepy
%pip install git+https://github.com/openai/CLIP.git



  Cloning https://github.com/openai/CLIP.git to /tmp/pip-req-build-ptqtmpgl
  Running command git clone --filter=blob:none --quiet https://github.com/openai/CLIP.git /tmp/pip-req-build-ptqtmpgl
  Resolved https://github.com/openai/CLIP.git to commit dcba3cb2e2827b402d2701e7e1c7d9fed8a20ef1
  Preparing metadata (setup.py) ... done


In [93]:
%pip install llama-index
%pip install -U openai-whisper
%pip install groq


In [94]:
%pip install lancedb
%pip install moviepy
%pip install pytube
%pip install pydub
%pip install SpeechRecognition
%pip install ffmpeg-python
%pip install soundfile
%pip install torch torchvision
%pip install matplotlib scikit-image
%pip install ftfy regex tqdm

ffmpeg-library enables you to use FFmpeg in Python to manipulate various media files for different purposes like building comprehensive multimedia applications, preprocessing media files.

MoviePy is a Python library for video editing, enabling cutting, concatenations, title insertions, video compositing, and effects like animations or color grading.

Pytube is a Python library used for downloading videos from YouTube. It supports downloading in various formats, resolutions, and also direct audio extraction.


Pydub is a Python library for audio manipulation, enabling easy loading,
editing, and exporting of audio files in various formats with minimal code.

The SpeechRecognition library in Python allows you to convert spoken language into text using various engines and APIs, such as Google Speech Recognition, IBM Speech to Text, etc.


SoundFile is a Python library for reading from and writing to audio files, supporting many formats through the libsndfile library, ideal for high-quality audio processing.

FTFY (Fix Text For You) is a Python library that fixes broken Unicode text and mojibake (garbled text due to encoding issues), making text legible again.

OpenAI Whisper is a robust, multilingual speech recognition model developed by OpenAI. It converts speech into text and supports various languages with high accuracy.

pprint is a Python module that provides a capability to "pretty-print" complex data structures in a well-formatted and more readable way than the basic print function.

In [95]:
from moviepy.editor import VideoFileClip
from pathlib import Path
import speech_recognition as sr
from pytube import YouTube
from pprint import pprint
from PIL import Image
import matplotlib.pyplot as plt

In [96]:
import os
from google.colab import userdata

GROQ_API_KEY = userdata.get("GROQ_API_KEY")
os.environ["GROQ_API_KEY"] = GROQ_API_KEY

In [97]:
import os
print(os.getcwd())

/content


In [98]:
video_url="https://youtu.be/3dhcmeOTZ_Q"

In [99]:
output_video_path = "/content/video_data/"

In [100]:
# from the video i am going to collect images,audio,text
output_folder = "/content/mixed_data/"
output_audio_path = "/content/mixed_data/output_audio.wav"

In [101]:
!mkdir mixed_data

mkdir: cannot create directory ‘mixed_data’: File exists


In [102]:
filepath=output_video_path + "input_vid.mp4"
print(filepath)

/content/video_data/input_vid.mp4


In [103]:
from pytube import YouTube
def download_video(url,output_path):
  yt = YouTube(url)
  metadata = {"Author": yt.author, "Title": yt.title, "Views": yt.views}

  yt.streams.get_highest_resolution().download(
        output_path=output_path, filename="input_vid.mp4"
    )
  return metadata

In [104]:
from moviepy.editor import VideoFileClip
def video_to_images(video_path,output_folder):
  clip=VideoFileClip(video_path)
  clip.write_images_sequence(
      os.path.join(output_folder,"frame%04d.png"),fps=0.2
  )

In [105]:
def video_to_audio(video_path,output_audio_path):
  clip=VideoFileClip(video_path)
  audio=clip.audio
  audio.write_audiofile(output_audio_path)

In [106]:
def audio_to_text(audio_path):
  recognizer=sr.Recognizer()
  audio=sr.AudioFile(audio_path)

  with audio as source:
    audio_data=recognizer.record(source)

    try:

      #recognize the speech
      text = recognizer.recognize_whisper(audio_data)

    except sr.UnknownValueError:
      print("Speech recognition could not understand the audio.")
  return text

In [107]:
video_url

'https://youtu.be/3dhcmeOTZ_Q'

In [108]:
output_video_path

'/content/video_data/'

In [109]:
import os

output_dir = "/content/video_data"
os.makedirs(output_dir, exist_ok=True)


In [110]:
import whisper

model = whisper.load_model("base")
result = model.transcribe(video_path)

transcript = result["text"]
print("✅ Transcript length:", len(transcript))





✅ Transcript length: 3396


In [111]:
%pip install -U yt-dlp


In [112]:
import cv2
import os

video_path = "/content/video_data/video.mp4"
frames_dir = "/content/video_data/frames"
os.makedirs(frames_dir, exist_ok=True)

cap = cv2.VideoCapture(video_path)
fps = int(cap.get(cv2.CAP_PROP_FPS))
frame_id = 0
saved = 0

while True:
    ret, frame = cap.read()
    if not ret:
        break

    if frame_id % fps == 0:  # 1 frame per second
        ts = frame_id // fps
        frame_path = f"{frames_dir}/frame_{ts:04d}.jpg"
        cv2.imwrite(frame_path, frame)
        saved += 1

    frame_id += 1

cap.release()
print(f"✅ Saved {saved} frames to {frames_dir}")


✅ Saved 242 frames to /content/video_data/frames


In [113]:
import os

assert os.path.exists(video_path), "❌ video.mp4 not found"
print("✅ Video file exists and is ready")


✅ Video file exists and is ready


In [114]:
import whisper

model = whisper.load_model("base")
result = model.transcribe(video_path)

transcript = result["text"]


In [115]:
print(type(image_files))
print(type(image_files[0]))


<class 'list'>
<class 'str'>


In [116]:
import glob
from llama_index.readers.file import ImageReader

frames_dir = "/content/video_data/frames"

# Get image paths
image_files = sorted(glob.glob(f"{frames_dir}/*.jpg"))

print(type(image_files), type(image_files[0]), len(image_files))



<class 'list'> <class 'str'> 242


In [117]:
from llama_index.core.schema import ImageDocument

image_docs = []

for img_path in image_files:
    image_docs.append(
        ImageDocument(
            image_path=img_path,
            metadata={"file_path": img_path}
        )
    )

print("✅ Image documents loaded:", len(image_docs))


✅ Image documents loaded: 242


In [118]:
from llama_index.core import Document

text_doc = Document(
    text=transcript,
    metadata={"type": "video_transcript"}
)

print("✅ text_doc created")


✅ text_doc created


In [119]:
from llama_index.core import Document
from llama_index.core.text_splitter import SentenceSplitter

splitter = SentenceSplitter(
    chunk_size=60,      # <= 77 tokens for CLIP
    chunk_overlap=10
)

text_chunks = splitter.split_text(transcript)

text_docs = [
    Document(
        text=chunk,
        metadata={"type": "video_transcript"}
    )
    for chunk in text_chunks
]

print("✅ Transcript chunks:", len(text_docs))


✅ Transcript chunks: 15


In [120]:
from llama_index.embeddings.clip import ClipEmbedding
from llama_index.vector_stores.lancedb import LanceDBVectorStore
from llama_index.core import StorageContext
from llama_index.core.indices.multi_modal import MultiModalVectorStoreIndex
import lancedb

embed_model = ClipEmbedding()

db = lancedb.connect("/content/lancedb")

vector_store = LanceDBVectorStore(
    uri="/content/lancedb",
    table_name="video_multimodal",
    embedding=embed_model
)

storage_context = StorageContext.from_defaults(vector_store=vector_store)


# 🔥 CRITICAL FIX: use text_docs (plural)
index = MultiModalVectorStoreIndex.from_documents(
    image_docs + text_docs,   # ✅ THIS FIXES EVERYTHING
    storage_context=storage_context,
    embed_model=embed_model
)

print("✅ Multimodal data stored in LanceDB successfully")


✅ Multimodal data stored in LanceDB successfully


In [121]:
db.table_names()

  db.table_names()



['video_multimodal']

In [122]:
table = db.open_table("video_multimodal")
table.count_rows()

30

In [122]:
from llama_index.core.indices.multi_modal import MultiModalVectorStoreIndex

# reuse existing vector_store and storage_context
index = MultiModalVectorStoreIndex.from_vector_store(
    vector_store=vector_store,
    embed_model=embed_model
)


In [123]:
index.insert_nodes(image_docs)

print("✅ Image documents inserted into LanceDB")


✅ Image documents inserted into LanceDB


In [124]:
table = db.open_table("video_multimodal")
table.count_rows()


30

In [125]:
table.to_pandas().head()


,id,doc_id,vector,text,metadata
0,0d63e6f2-2e44-4b8e-8c07-8f2e0671e349,d6e39df4-b207-48a2-8a49-abfb570c4a0f,"[-0.1048584, 0.121398926, 0.006954193, 0.03619...",Lenny regression is a statistical technique fo...,"{'_node_content': '{""id_"": ""0d63e6f2-2e44-4b8e..."
1,27ae0b43-db7c-447c-b380-f31a4aa2601b,3021e9e1-a83d-4806-aa77-7b76e9ad95c2,"[-0.21691895, 0.22729492, -0.13061523, 0.08843...","In layman's terms, think of it as fitting a li...","{'_node_content': '{""id_"": ""27ae0b43-db7c-447c..."
2,89788dba-ab5a-43e2-afa5-152422ec98ef,1ddefc65-bb73-4c05-b788-3f6936a67d33,"[-0.3305664, 0.18383789, -0.171875, -0.0816650...",You might be familiar with the linear function...,"{'_node_content': '{""id_"": ""89788dba-ab5a-43e2..."
3,2f8b9268-fd94-4724-b38b-202dc2333a1e,ebc913ad-734b-4ef9-b1ef-8ebbd84042cb,"[0.008117676, 0.3076172, 0.18688965, -0.114013...","x on the other hand, would serve as the input ...","{'_node_content': '{""id_"": ""2f8b9268-fd94-4724..."
4,66acf0c1-7db2-47c8-9a0f-26465f9fd480,73790b19-fac4-4dcd-b288-18ed18f3943f,"[0.09313965, 0.13952637, -0.1496582, -0.039855...",The m or beta 1 coefficient controls the slope...,"{'_node_content': '{""id_"": ""66acf0c1-7db2-47c8..."


In [126]:
from llama_index.core.indices.multi_modal import MultiModalVectorStoreIndex

index = MultiModalVectorStoreIndex.from_vector_store(
    vector_store=vector_store,
    embed_model=embed_model
)

print("✅ Index loaded from existing LanceDB table")


✅ Index loaded from existing LanceDB table


In [127]:
index.insert_nodes(image_docs)

print("✅ Image documents inserted into LanceDB")

✅ Image documents inserted into LanceDB


In [128]:
table = db.open_table("video_multimodal")
table.count_rows()


30

In [129]:
len(image_docs)


242

In [130]:
index.insert_nodes(image_docs)

print("✅ Image insertion command completed")


✅ Image insertion command completed


In [131]:
table = db.open_table("video_multimodal")
table.count_rows()


30

In [132]:
from llama_index.core.schema import ImageDocument
import glob

frames_dir = "/content/video_data/frames"
image_files = sorted(glob.glob(f"{frames_dir}/*.jpg"))

print("Frame files found:", len(image_files))  # MUST be 242

image_docs = [
    ImageDocument(
        image_path=img_path,
        metadata={"file_path": img_path}
    )
    for img_path in image_files
]

print("Image docs rebuilt:", len(image_docs))  # MUST be 242


Frame files found: 242
Image docs rebuilt: 242


In [133]:
index.insert_nodes(image_docs)

print("✅ All 242 image documents inserted into LanceDB")


✅ All 242 image documents inserted into LanceDB


In [134]:
df=table.to_pandas()
df[df["text"].isna()].shape

(0, 5)

In [135]:
from llama_index.core.indices.multi_modal import MultiModalVectorStoreIndex

index = MultiModalVectorStoreIndex.from_vector_store(
    vector_store=vector_store,
    embed_model=embed_model
)

print("✅ Index loaded from LanceDB")


✅ Index loaded from LanceDB


In [136]:
query = "Explain linear regression visually"

retriever = index.as_retriever(similarity_top_k=6)
nodes = retriever.retrieve(query)

print("✅ Retrieved nodes:", len(nodes))


✅ Retrieved nodes: 3


In [137]:
context = []
retrieved_images = []

for node in nodes:
    # text chunks
    if node.text and node.text.strip():
        context.append(node.text)

    # image frames
    file_path = node.metadata.get("file_path")
    if file_path:
        retrieved_images.append(file_path)

print("Text chunks:", len(context))
print("Image frames:", len(retrieved_images))


Text chunks: 3
Image frames: 0


In [138]:
retriever = index.as_retriever(
    similarity_top_k=6,
    image_similarity_top_k=4   # 🔥 THIS enables image retrieval
)

nodes = retriever.retrieve(query)

print("Retrieved nodes:", len(nodes))


Retrieved nodes: 3


In [139]:
retriever = index.as_retriever(
    similarity_top_k=3,
    image_similarity_top_k=3
)

nodes = retriever.retrieve(query)

text_nodes = [n for n in nodes if n.text and n.text.strip()]
image_nodes = [n for n in nodes if n.metadata.get("file_path")]

print("Text nodes:", len(text_nodes))
print("Image nodes:", len(image_nodes))


Text nodes: 2
Image nodes: 0


In [140]:
query = "scatter plot with regression line"
# or
query = "graph showing data points and a straight line"
# or
query = "visual explanation of linear regression"


In [141]:
nodes = retriever.retrieve(query)

text_nodes = [n for n in nodes if n.text and n.text.strip()]
image_nodes = [n for n in nodes if n.metadata.get("file_path")]

print("Text nodes:", len(text_nodes))
print("Image nodes:", len(image_nodes))


Text nodes: 2
Image nodes: 0


In [142]:
image_query = "scatter plot regression line graph"

image_retriever = index.as_retriever(image_similarity_top_k=4)
image_nodes = image_retriever.retrieve(image_query)

print("Text nodes:", len(text_nodes))
print("Image nodes:", len(image_nodes))

Text nodes: 2
Image nodes: 1


In [143]:
query = "scatter plot with regression line"


In [144]:
retriever = index.as_retriever(
    similarity_top_k=3,
    image_similarity_top_k=3
)

nodes = retriever.retrieve(query)

In [145]:
text_nodes = []
image_nodes = []

for n in nodes:
    if n.text and n.text.strip():
        text_nodes.append(n)

    if n.metadata.get("file_path"):
        image_nodes.append(n)

print("Text nodes:", len(text_nodes))
print("Image nodes:", len(image_nodes))


Text nodes: 2
Image nodes: 0


In [146]:
for i, n in enumerate(text_nodes, 1):
    print(f"\n--- Text Chunk {i} ---")
    print(n.text)



--- Text Chunk 1 ---
To validate a linear regression, there are a number of techniques. Machine learning practitioners will often take a third of the data and put it into the test data set. The remaining two thirds will become the training data set.

--- Text Chunk 2 ---
The training data set will then be used to fit the regression line. The test data set will then be used to validate the regression line. This is done to make sure that the regression performs well on data it has not seen before.
